# Behavioral Monitoring — Dev Log

## Objetivo

Detecção de drift real via teste de Kolmogorov-Smirnov de duas amostras
(`scipy.stats.ks_2samp`) — compara qualquer métrica numérica de dois
períodos e mede se a distribuição mudou estatisticamente.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.behavioral_monitoring.drift import detect_drift
from core.policy_engine.engine import evaluate
from core.trust_score.scorer import compute_trust_score
from shared.schemas import DataCategory, LegalBasis, PIIDetectionResult

neutral = PIIDetectionResult(findings=[], has_sensitive_data=False, summary="sem PII")
baseline = []
for _ in range(10):
    d = evaluate(data_categories=[DataCategory.PERSONAL], legal_basis=LegalBasis.LEGITIMATE_INTEREST)
    baseline.append(compute_trust_score(pii_result=neutral, policy_decisions=d).score)
current = []
for _ in range(10):
    d = evaluate(
        data_categories=[DataCategory.SENSITIVE], legal_basis=LegalBasis.NOT_DETERMINED,
        context={"data_subtype": "biometric", "automated_decision": True, "human_review": False},
    )
    current.append(compute_trust_score(pii_result=neutral, policy_decisions=d).score)

result = detect_drift(baseline, current, metric_name="trust_score")
print(f"baseline={baseline[:3]}... | current={current[:3]}...")
print(result.summary)

baseline=[100.0, 100.0, 100.0]... | current=[5.0, 5.0, 5.0]...
Drift DETECTADO em 'trust_score': estatística KS=1.0000, p-valor=1.083e-05 < alpha=0.05 — as distribuições baseline/atual são estatisticamente diferentes.


## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/behavioral_monitoring/tests -v
```

6/6 testes passando, incluindo drift medido sobre `trust_score.score` real.